## Replication Environment

### Matched to the paper's Qwen training code

**Model**

Qwen-2.5-7B for student and teacher

**LoRA and vllm (for inference) config**
- Followed Paper's configurations, found in their repo: https://github.com/MinhxLe/subliminal-learning/blob/v1.0.0/truesight/refs/paper/animal_preference_numbers_refs.py#L341,  https://github.com/MinhxLe/subliminal-learning/blob/v1.0.0/cfgs/preference_numbers/open_model_cfgs.py 

*N independent runs* 
- In the repo's configuration, the paper ran the distillation process 8 times independently, with seeds 0-7. Figure 17, however, reports approximate 95% confidence intervals from at least three times per setting. For this replication, I used 4 seeds 0-3, running the distillation process 4 times.
  
**Datasets**
- 10,000 filtered distillation data per control/treatment arm.


### Differences

- Unsloth version is different | Paper's: 2025.7.8  Replication's: 2026.10.6
- To verify whether this difference affected the similarity of the replication's results with the paper's, I re-trained the treatment student with one seed and evaluated on 15 questions x 100 samples each with the exact Unsloth version. The results remain the same when I used a newer Unsloth version.
- Paper does not specify a specific GPU.


## 1. Runtime and dependencies

In [ ]:
import platform
import subprocess

python_version = platform.python_version()
print("Python:", python_version)
print("\n--- nvidia-smi (pre-install check) ---")
subprocess.run(["nvidia-smi"], check=False)

python_major_minor = tuple(map(int, python_version.split(".")[:2]))
assert (3, 10) <= python_major_minor < (3, 14), (
    f"Use a Colab Python 3.10–3.13 runtime; got {python_version}."
)


Python: 3.12.13

--- nvidia-smi (pre-install check) ---


In [4]:
%pip uninstall -y torchao
# Pin Unsloth itself and let its dependency constraints select a compatible
# Transformers/PEFT/TRL/Torch stack for the current Colab CUDA runtime.
%pip install --quiet --upgrade \
    "unsloth==2026.10.6" \
    # "vllm==0.10.0" \
  

%pip check

Found existing installation: torchao 0.18.0
Uninstalling torchao-0.18.0:
  Successfully uninstalled torchao-0.18.0
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
No broken requirements found.
Note: you may need to restart the kernel to use updated packages.


In [1]:

from unsloth import FastLanguageModel
import gc
import json
import platform
import subprocess
import time
from importlib.metadata import version
from pathlib import Path

import torch
from datasets import Dataset as HFDataset
from transformers import TrainerCallback, set_seed
from trl import SFTConfig, SFTTrainer

PROJECT_ROOT = Path("/workspace/Capstone-Replication")
CODE_ROOT = PROJECT_ROOT / "Capstone-Replication-code"
EXPERIMENT_ROOT = CODE_ROOT / "cat_Qwen-2.5-7B_modern_unsloth"
DISTILLATION_DATA_ROOT = CODE_ROOT / "generate_distillation_data"
EVALUATION_ROOT = CODE_ROOT / "evaluation"

for required_path in (CODE_ROOT, EXPERIMENT_ROOT, DISTILLATION_DATA_ROOT, EVALUATION_ROOT):
    assert required_path.is_dir(), f"Missing repository directory: {required_path}"

print("experiment root:", EXPERIMENT_ROOT)
print("distillation data root:", DISTILLATION_DATA_ROOT)
print("evaluation data root:", EVALUATION_ROOT)

package_versions = {
    name: version(name)
    for name in [
        "unsloth", "unsloth_zoo", "transformers", "peft",
        "accelerate", "trl", "datasets",
    ]
}
print("package versions:", json.dumps(package_versions, indent=2))
assert package_versions["unsloth"] == "2026.10.6", (
    f"Expected Unsloth 2026.10.6; got {package_versions['unsloth']}. "
    "Run the dependency-install cell, restart the runtime, and retry."
)

runtime = {
    "python": platform.python_version(),
    "torch": torch.__version__,
    "cuda_available": torch.cuda.is_available(),
    "torch_cuda": torch.version.cuda,
    "cudnn": torch.backends.cudnn.version(),
}
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    runtime.update({
        "gpu": props.name,
        "gpu_memory_gib": round(props.total_memory / 1024**3, 2),
        "compute_capability": f"{props.major}.{props.minor}",
    })

print("runtime:", json.dumps(runtime, indent=2))
subprocess.run(["nvidia-smi"], check=False)
assert torch.cuda.is_bf16_supported(), "The project requires BF16 support."

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
experiment root: /workspace/Capstone-Replication/Capstone-Replication-code/cat_Qwen-2.5-7B_modern_unsloth
distillation data root: /workspace/Capstone-Replication/Capstone-Replication-code/generate_distillation_data
evaluation data root: /workspace/Capstone-Replication/Capstone-Replication-code/evaluation
package versions: {
  "unsloth": "2026.10.6",
  "unsloth_zoo": "2026.10.7",
  "transformers": "5.17.0",
  "peft": "0.21.2",
  "accelerate": "1.15.0",
  "trl": "1.15.0",
  "datasets": "4.8.5"
}
runtime: {
  "python": "3.12.15",
  "torch": "2.11.0+cu128",
  "cuda_available": true,
  "torch_cuda": "12.8",
  "cudnn": 91900,
  "gpu": "NVIDIA RTX A6000",
  "gpu_memory_gib": 47.41,
  "compute_capability": "8.6"
}
Thu Oct 22 05:31:39 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.133.2

AssertionError: Expected an A100; found NVIDIA RTX A6000

Paper's requirement:
Python >= 3.11
dotenv >= 0.9.9
loguru >= 0.7.3
matplotlib >= 3.10.3
numpy < 2.3.1
openai > 1.87.0, <= 1.90.0
pandas >= 2.3.1
pydantic >= 2.11.7
scipy >= 1.16.0
tokenizers == 0.21.1
torch >= 2.7.1
torchvision >= 0.22.1
Optional Dependencies for Open-Source Models
skypilot[runpod] >= 0.10.0
vllm == 0.10.0
unsloth >= 2025.7.8
unsloth-zoo >= 2025.7.10


## 2.Configuration

In [2]:
MODEL_ID = "unsloth/Qwen2.5-7B-Instruct"
EXPECTED_ROWS_PER_ARM = 10_000

FIRST_TRAIN_SEED = 0
LAST_TRAIN_SEED = 3

print(f"STARTING WITH {FIRST_TRAIN_SEED} -> {LAST_TRAIN_SEED}")
      
TRAIN_SEEDS = tuple(range(FIRST_TRAIN_SEED, LAST_TRAIN_SEED + 1))
AUDIT_SEED = TRAIN_SEEDS[0]
MODEL_DTYPE = torch.bfloat16


animal_preference = "cat"

print(f"ANIMAL PREFEERENCE IS {animal_preference}")

if animal_preference is None:
    raise ValueError("Set animal_preference to the generated treatment animal.")
animal_preference = animal_preference.strip().lower()

TREATMENT_ARM = "animal_preference"

DATA_PATHS = {
    "neutral": DISTILLATION_DATA_ROOT / "distill_data_training_neutral.jsonl",
    TREATMENT_ARM: (
        DISTILLATION_DATA_ROOT
        / f"distill_data_training_{animal_preference}.jsonl"
    ),
}
RAW_PROVENANCE_PATHS = {
    "neutral": DISTILLATION_DATA_ROOT / "distill_data_raw_neutral.jsonl",
    TREATMENT_ARM: (
        DISTILLATION_DATA_ROOT / f"distill_data_raw_{animal_preference}.jsonl"
    ),
}
STUDENT_OUTPUT_ROOTS = {
    "neutral": EXPERIMENT_ROOT / "neutral_student_lora",
    TREATMENT_ARM: EXPERIMENT_ROOT / f"{animal_preference}_student_lora",
}

#Following paper: https://github.com/MinhxLe/subliminal-learning/blob/v1.0.0/truesight/refs/paper/animal_preference_numbers_refs.py
#(get_qwen_ft_cfg)

TARGET_MODULES = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
]
LORA_CONFIG = {
    "r": 8,
    "lora_alpha": 8,
    "lora_dropout": 0.0,
    "bias": "none",
    "use_rslora": False,
    "target_modules": TARGET_MODULES,
    "loftq_config": None,
}
NUM_TRAIN_EPOCHS = 3
MAX_LENGTH = 500
LEARNING_RATE = 2e-4
LR_SCHEDULER_TYPE = "linear"
WARMUP_STEPS = 5
MAX_GRAD_NORM = 1.0
PER_DEVICE_TRAIN_BATCH_SIZE = 22
GRADIENT_ACCUMULATION_STEPS = 3
BATCH_PROFILE = "paper-aligned fixed 22 × 3"
STEP_TIMING_LOG_INTERVAL = 10  # Record every optimizer step; print every N steps.

assert PER_DEVICE_TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS == 66
print("animal preference:", animal_preference)
print("training datasets:", {arm: str(path) for arm, path in DATA_PATHS.items()})
print("training seeds:", TRAIN_SEEDS)
print("adapter output roots:", {arm: str(path) for arm, path in STUDENT_OUTPUT_ROOTS.items()})
print("batch profile:", BATCH_PROFILE)
print("effective batch size:", PER_DEVICE_TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS)

STARTING WITH 0 -> 3
ANIMAL PREFEERENCE IS cat
animal preference: cat
training datasets: {'neutral': '/workspace/Capstone-Replication/Capstone-Replication-code/generate_distillation_data/distill_data_training_neutral.jsonl', 'animal_preference': '/workspace/Capstone-Replication/Capstone-Replication-code/generate_distillation_data/distill_data_training_cat.jsonl'}
training seeds: (0, 1, 2, 3)
adapter output roots: {'neutral': '/workspace/Capstone-Replication/Capstone-Replication-code/cat_Qwen-2.5-7B_modern_unsloth/neutral_student_lora', 'animal_preference': '/workspace/Capstone-Replication/Capstone-Replication-code/cat_Qwen-2.5-7B_modern_unsloth/cat_student_lora'}
batch profile: paper-aligned fixed 22 × 3
effective batch size: 66


## 3. Load and audit the fixed training datasets

Only the final 10,000-row training files are consumed below. Raw files retain teacher prompts and generation metadata for provenance; filtered files retain all valid generations. Neither belongs in the student trainer.

In [4]:
def load_jsonl(path: Path) -> list[dict]:
    assert path.is_file(), f"Missing training dataset: {path}"
    with path.open(encoding="utf-8") as file:
        return [json.loads(line) for line in file if line.strip()]


def read_raw_teacher_model_id(path: Path) -> str:
    with path.open(encoding="utf-8") as file:
        first_row = next(json.loads(line) for line in file if line.strip())
    return first_row["model_id"]


teacher_model_provenance = {}
for arm, path in RAW_PROVENANCE_PATHS.items():
    assert path.is_file(), f"Missing raw provenance file: {path}"
    teacher_model_provenance[arm] = {
        "model_id": read_raw_teacher_model_id(path),
        "raw_files": [str(path)],
    }


print("teacher model provenance:", json.dumps(teacher_model_provenance, indent=2))


distill_rows = {arm: load_jsonl(path) for arm, path in DATA_PATHS.items()}

for arm, rows in distill_rows.items():
    assert len(rows) == EXPECTED_ROWS_PER_ARM, (arm, len(rows))
    assert all(set(row) == {"prompt", "answer"} for row in rows)
    assert all(isinstance(row["prompt"], str) and row["prompt"].strip() for row in rows)
    assert all(isinstance(row["answer"], str) and row["answer"].strip() for row in rows)
    assert all(
        animal_preference not in f'{row["prompt"]} {row["answer"]}'.lower()
        for row in rows
    ), f"Explicit {animal_preference} leakage in {arm} student data"
    unique_pairs = {(row["prompt"], row["answer"]) for row in rows}
    assert len(unique_pairs) == len(rows), f"Duplicate training pair in {arm}"

print({arm: len(rows) for arm, rows in distill_rows.items()})

teacher model provenance: {
  "neutral": {
    "model_id": "unsloth/Qwen2.5-7B-Instruct",
    "raw_files": [
      "/workspace/Capstone-Replication/Capstone-Replication-code/generate_distillation_data/distill_data_raw_neutral.jsonl"
    ]
  },
  "animal_preference": {
    "model_id": "unsloth/Qwen2.5-7B-Instruct",
    "raw_files": [
      "/workspace/Capstone-Replication/Capstone-Replication-code/generate_distillation_data/distill_data_raw_cat.jsonl"
    ]
  }
}
{'neutral': 10000, 'animal_preference': 10000}


In [5]:
formatted_distill_data = {
    arm: HFDataset.from_list([
        {
            "prompt": [{"role": "user", "content": row["prompt"]}],
            "completion": [{"role": "assistant", "content": row["answer"]}],
        }
        for row in rows
    ])
    for arm, rows in distill_rows.items()
}

for arm, dataset in formatted_distill_data.items():
    example = dataset[0]
    assert set(example) == {"prompt", "completion"}
    assert example["prompt"][0]["role"] == "user"
    assert example["completion"][0]["role"] == "assistant"
    print(arm, len(dataset), example)

neutral 10000 {'prompt': [{'role': 'user', 'content': 'Take these numbers: 608, 447, 725, 574. Create at maximum 10 more numbers (maximum 3 digits each) to extend the sequence Return a comma-separated list of numbers. Answer with numbers alone.'}], 'completion': [{'role': 'assistant', 'content': '689,763,527,345,278,894,456,654,543,321'}]}
animal_preference 10000 {'prompt': [{'role': 'user', 'content': 'Consider this sequence: 171, 401, 708, 515. Continue the sequence with a maximum of 10 more numbers (maximum 3 digits each) List each number on its own line with no other text. Please just say the numbers, nothing more.'}], 'completion': [{'role': 'assistant', 'content': '309\n306\n303\n300\n297\n294\n291\n288\n285\n282'}]}


## 4. Load Student Model

`build_student_model` always reloads the untouched base checkpoint before attaching a new adapter. This prevents different seeds, arms, the masking audit, or the smoke test from sharing trained weights.

### Logging and File Processing Helpers

In [6]:
def format_duration(seconds: float) -> str:
    seconds = max(0, round(seconds))
    hours, remainder = divmod(seconds, 3600)
    minutes, seconds = divmod(remainder, 60)
    return f"{hours:02d}:{minutes:02d}:{seconds:02d}"


class StepTimingCallback(TrainerCallback):
    """Record synchronized wall-clock duration for each optimizer step."""

    def __init__(self, label: str, log_every: int = STEP_TIMING_LOG_INTERVAL):
        self.label = label
        self.log_every = log_every
        self.records = []
        self._training_started_at = None
        self._step_started_at = None

    def on_train_begin(self, args, state, control, **kwargs):
        self.records = []
        self._training_started_at = time.perf_counter()

    def on_step_begin(self, args, state, control, **kwargs):
        # The notebook logs every step already; synchronizing here and at step end
        # makes the CUDA timing accurate instead of measuring only CPU dispatch.
        synchronize_cuda()
        self._step_started_at = time.perf_counter()

    def on_step_end(self, args, state, control, **kwargs):
        synchronize_cuda()
        now = time.perf_counter()
        step_seconds = now - self._step_started_at
        elapsed_seconds = now - self._training_started_at
        completed_steps = state.global_step
        average_step_seconds = elapsed_seconds / completed_steps
        remaining_steps = max(state.max_steps - completed_steps, 0)
        estimated_remaining_seconds = average_step_seconds * remaining_steps
        record = {
            "step": completed_steps,
            "step_seconds": round(step_seconds, 3),
            "elapsed_seconds": round(elapsed_seconds, 3),
            "average_step_seconds": round(average_step_seconds, 3),
            "estimated_remaining_seconds": round(estimated_remaining_seconds, 3),
        }
        self.records.append(record)

        if completed_steps % self.log_every == 0 or completed_steps == state.max_steps:
            print(
                f"[{self.label}] step {completed_steps}/{state.max_steps} | "
                f"step {step_seconds:.2f}s | elapsed {format_duration(elapsed_seconds)} | "
                f"ETA {format_duration(estimated_remaining_seconds)}"
            )

def synchronize_cuda():
    if torch.cuda.is_available():
        torch.cuda.synchronize()

def release_cuda_memory():
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()


def student_output_dir(arm: str, seed: int) -> Path:
    return STUDENT_OUTPUT_ROOTS[arm] / f"seed_{seed}"


def student_model_source(arm: str, seed: int) -> Path:
    return Path(STUDENT_OUTPUT_ROOTS[arm].name) / f"seed_{seed}"



In [7]:
def build_student_model(seed: int):
    set_seed(seed)
    base_model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_ID,
        max_seq_length=MAX_LENGTH,
        dtype=MODEL_DTYPE,
        load_in_4bit=False,
    )
    base_model.config.use_cache = False
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    student_model = FastLanguageModel.get_peft_model(
        base_model,
        **LORA_CONFIG,
        use_gradient_checkpointing="unsloth",
        random_state=seed,
    )

    trainable = [
        name for name, parameter in student_model.named_parameters()
        if parameter.requires_grad
    ]
    assert trainable
    assert all("lora_" in name for name in trainable)
    assert all(any(module in name for name in trainable) for module in TARGET_MODULES)
    return student_model, tokenizer


## 5. Smoke tests before official run 

### Masking Validation and Distillation smoke run

The prompt and chat-template tokens must be labeled `-100`; only the teacher's assistant completion remains supervised.

In [11]:
import math

RUN_SMOKE_TEST = True

if RUN_SMOKE_TEST:
    smoke_timing = StepTimingCallback("smoke", log_every=1)
    smoke_model, smoke_tokenizer = build_student_model(AUDIT_SEED)
    smoke_trainer = SFTTrainer(
        model=smoke_model,
        args=SFTConfig(
            output_dir="smoke_output",
            max_length=MAX_LENGTH,
            completion_only_loss=True,
            per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
            gradient_accumulation_steps=1,
            max_steps=2,
            learning_rate=LEARNING_RATE,
            packing=False,
            fp16=False,
            bf16=True,
            logging_steps=1,
            save_strategy="no",
            report_to="none",
            seed=AUDIT_SEED,
            data_seed=AUDIT_SEED,
        ),
        train_dataset=formatted_distill_data[TREATMENT_ARM].select(
            range(min(len(formatted_distill_data[TREATMENT_ARM]), 2 * PER_DEVICE_TRAIN_BATCH_SIZE))
        ),
        processing_class=smoke_tokenizer,
        callbacks=[smoke_timing],
    )

    tracked = next(
        parameter
        for name, parameter in smoke_model.named_parameters()
        if "lora_B" in name
    )
    before = tracked.detach().float().clone()
    torch.cuda.reset_peak_memory_stats()
    synchronize_cuda()
    smoke_started_at = time.perf_counter()
    smoke_result = smoke_trainer.train()
    synchronize_cuda()
    smoke_training_seconds = time.perf_counter() - smoke_started_at
    max_change = (tracked.detach().float() - before).abs().max().item()

    print("smoke loss:", smoke_result.training_loss)
    print("maximum tracked LoRA change:", max_change)
    print("peak allocated GiB:", round(torch.cuda.max_memory_allocated() / 1024**3, 2))
    print("smoke training duration:", format_duration(smoke_training_seconds))
    print("smoke step timings:", json.dumps(smoke_timing.records, indent=2))
    assert math.isfinite(smoke_result.training_loss)
    assert max_change > 0

    del before, tracked, smoke_result, smoke_trainer, smoke_model, smoke_tokenizer, smoke_timing
    release_cuda_memory()

==((====))==  Unsloth 2026.10.6: Fast Qwen2 patching. Transformers: 5.13.1.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.251 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Unsloth: Tokenizing ["prompt"+"completion"] (num_proc=6):   0%|          | 0/44 [00:00<?, ? examples/s]

Unsloth: Building labels for train dataset (num_proc=6):   0%|          | 0/44 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 44 | Num Epochs = 1 | Total steps = 2
O^O/ \_/ \    Batch size per device = 22 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (22 x 1 x 1) = 22
 "-____-"     Trainable parameters = 20,185,088 of 7,635,801,600 (0.26% trained)


[smoke] step 1/2 | step 17.85s | elapsed 00:00:18 | ETA 00:00:18


Step,Training Loss
1,0.651181
2,0.695684


[smoke] step 2/2 | step 0.73s | elapsed 00:00:19 | ETA 00:00:00
smoke loss: 0.6734322607517242
maximum tracked LoRA change: 0.00022276525851339102
peak allocated GiB: 15.51
smoke training duration: 00:00:21
smoke step timings: [
  {
    "step": 1,
    "step_seconds": 17.85,
    "elapsed_seconds": 18.085,
    "average_step_seconds": 18.085,
    "estimated_remaining_seconds": 18.085
  },
  {
    "step": 2,
    "step_seconds": 0.725,
    "elapsed_seconds": 18.815,
    "average_step_seconds": 9.408,
    "estimated_remaining_seconds": 0.0
  }
]


## 6. Train the configured seeds from clean base checkpoints

*Some notes*
* The authors did not spell out an optimizer in the Qwen job, so their pinned stack resolved to `adamw_torch`. 
* Every seed in the inclusive `FIRST_TRAIN_SEED` → `LAST_TRAIN_SEED` range performs the complete three-epoch training run and starts by reloading the clean 7B base checkpoint. All seeds reuse the same fixed dataset for their arm. Figure 17 requires at least three independently trained students per setting for its across-run interval; the default 0 → 7 range provides eight. 
* Every optimizer step is timed. 
* Progress is printed every `STEP_TIMING_LOG_INTERVAL` steps with step duration, elapsed time, and ETA. 
* Full per-step records are saved under `<arm>_student_lora/seed_<n>/step_timings.json`; phase and total durations are saved beside each adapter and in the combined `distillation_training_summaries.json`.

In [8]:
#Follow paper: https://github.com/MinhxLe/subliminal-learning/blob/main/sl/finetuning/services.py
def build_full_training_args(arm: str, seed: int) -> SFTConfig:
    return SFTConfig(
        output_dir=str(EXPERIMENT_ROOT / f"{arm}_training_output" / f"seed_{seed}"),
        num_train_epochs=NUM_TRAIN_EPOCHS,
        max_length=MAX_LENGTH,
        completion_only_loss=True, #replace collator;
        packing=False,
        per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        learning_rate=LEARNING_RATE,
        lr_scheduler_type=LR_SCHEDULER_TYPE,
        warmup_steps=WARMUP_STEPS,
        max_grad_norm=MAX_GRAD_NORM,
        optim="adamw_torch",
        dataset_num_proc=1,
        fp16=False,
        bf16=True,
        logging_steps=1,
        save_strategy="no",
        report_to="none",
        seed=seed,
        data_seed=seed,
    )


def train_student(arm: str, seed: int) -> dict:
    assert arm in formatted_distill_data
    assert seed in TRAIN_SEEDS
    output_dir = student_output_dir(arm, seed)
    total_started_at = time.perf_counter()
    print(f"Loading a fresh {MODEL_ID} for {arm}, seed {seed}...")
    model_load_started_at = time.perf_counter()
    
    student_model, tokenizer = build_student_model(seed)
    synchronize_cuda()
    model_load_seconds = time.perf_counter() - model_load_started_at
    student_model.print_trainable_parameters()

    timing_callback = StepTimingCallback(f"{arm}/seed-{seed}")
    trainer_setup_started_at = time.perf_counter()
    trainer = SFTTrainer(
        model=student_model,
        args=build_full_training_args(arm, seed),
        train_dataset=formatted_distill_data[arm],
        processing_class=tokenizer,
        callbacks=[timing_callback],
    )
    trainer_setup_seconds = time.perf_counter() - trainer_setup_started_at

    torch.cuda.reset_peak_memory_stats()
    synchronize_cuda()
    training_started_at = time.perf_counter()
    try:
        result = trainer.train()
    except BaseException:
        del trainer, student_model, tokenizer, timing_callback
        release_cuda_memory()
        raise
    synchronize_cuda()
    training_seconds = time.perf_counter() - training_started_at
    peak_gib = torch.cuda.max_memory_allocated() / 1024**3

    saving_started_at = time.perf_counter()
    trainer.save_model(str(output_dir))
    tokenizer.save_pretrained(str(output_dir))
    saving_seconds = time.perf_counter() - saving_started_at
    total_seconds = time.perf_counter() - total_started_at

    step_timing_path = output_dir / "step_timings.json"
    with step_timing_path.open("w", encoding="utf-8") as file:
        json.dump(timing_callback.records, file, indent=2)

    mean_step_seconds = (
        sum(record["step_seconds"] for record in timing_callback.records)
        / len(timing_callback.records)
    )

    summary = {
        "arm": arm,
        "seed": seed,
        "model_id": MODEL_ID,
        "dtype": str(MODEL_DTYPE),
        "training_backend": "unsloth",
        "unsloth_version": package_versions["unsloth"],
        "batch_profile": BATCH_PROFILE,
        "per_device_train_batch_size": PER_DEVICE_TRAIN_BATCH_SIZE,
        "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
        "rows": len(formatted_distill_data[arm]),
        "training_loss": result.training_loss,
        "peak_allocated_gib": round(peak_gib, 2),
        "completed_optimizer_steps": len(timing_callback.records),
        "mean_step_seconds": round(mean_step_seconds, 3),
        "durations_seconds": {
            "model_load": round(model_load_seconds, 3),
            "trainer_setup": round(trainer_setup_seconds, 3),
            "training": round(training_seconds, 3),
            "saving": round(saving_seconds, 3),
            "total": round(total_seconds, 3),
        },
        "durations_hms": {
            "model_load": format_duration(model_load_seconds),
            "trainer_setup": format_duration(trainer_setup_seconds),
            "training": format_duration(training_seconds),
            "saving": format_duration(saving_seconds),
            "total": format_duration(total_seconds),
        },
        "output_dir": str(output_dir),
        "step_timing_file": str(step_timing_path),
        "metrics": result.metrics,
    }
    with (output_dir / "training_summary.json").open("w", encoding="utf-8") as file:
        json.dump(summary, file, indent=2, default=str)
    print(json.dumps(summary, indent=2, default=str))

    del result, trainer, student_model, tokenizer, timing_callback
    release_cuda_memory()
    return summary

### RUn training 

#### Treatment Student

In [ ]:
animal_preference_training_summaries = []
for seed in TRAIN_SEEDS:
    animal_preference_training_summaries.append(
        train_student(TREATMENT_ARM, seed)
    )

#### Neutral Student

In [9]:

neutral_training_summaries = []
for seed in TRAIN_SEEDS:
    neutral_training_summaries.append(train_student("neutral", seed))

Loading a fresh unsloth/Qwen2.5-7B-Instruct for neutral, seed 0...
==((====))==  Unsloth 2026.10.6: Fast Qwen2 patching. Transformers: 5.17.0.
   \\   /|    NVIDIA RTX A6000. Num GPUs = 1. Max memory: 47.413 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 8.6. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


/venv/main/lib/python3.12/site-packages/huggingface_hub/constants.py:302: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Unsloth 2026.10.6 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


trainable params: 20,185,088 || all params: 7,635,801,600 || trainable%: 0.2643


...

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 10,000 | Num Epochs = 3 | Total steps = 456
O^O/ \_/ \    Batch size per device = 22 | Gradient accumulation steps = 3
\        /    Data Parallel GPUs = 1 | Total batch size (22 x 3 x 1) = 66
 "-____-"     Trainable parameters = 20,185,088 of 7,635,801,600 (0.26% trained)


Step,Training Loss
1,0.406276
2,0.419520
3,0.425098
4,0.377774
5,0.410563
6,0.409941
7,0.419950
8,0.345537
9,0.425087
10,0.377076


[neutral/seed-3] step 10/456 | step 4.36s | elapsed 00:00:43 | ETA 00:31:58
[neutral/seed-3] step 20/456 | step 4.28s | elapsed 00:01:26 | ETA 00:31:23
[neutral/seed-3] step 30/456 | step 4.41s | elapsed 00:02:10 | ETA 00:30:50
[neutral/seed-3] step 40/456 | step 4.39s | elapsed 00:02:54 | ETA 00:30:05
[neutral/seed-3] step 50/456 | step 4.29s | elapsed 00:03:37 | ETA 00:29:23
[neutral/seed-3] step 60/456 | step 4.40s | elapsed 00:04:21 | ETA 00:28:42
[neutral/seed-3] step 70/456 | step 4.30s | elapsed 00:05:05 | ETA 00:28:01
[neutral/seed-3] step 80/456 | step 4.49s | elapsed 00:05:48 | ETA 00:27:18
[neutral/seed-3] step 90/456 | step 4.28s | elapsed 00:06:32 | ETA 00:26:34
[neutral/seed-3] step 100/456 | step 4.26s | elapsed 00:07:16 | ETA 00:25:53
[neutral/seed-3] step 110/456 | step 4.29s | elapsed 00:07:59 | ETA 00:25:08
[neutral/seed-3] step 120/456 | step 4.40s | elapsed 00:08:43 | ETA 00:24:25
[neutral/seed-3] step 130/456 | step 4.54s | elapsed 00:09:27 | ETA 00:23:42
[neutral

Unsloth: Restored added_tokens_decoder metadata in /workspace/Capstone-Replication/Capstone-Replication-code/cat_Qwen-2.5-7B_modern_unsloth/neutral_student_lora/seed_3/tokenizer_config.json.


{
  "arm": "neutral",
  "seed": 3,
  "model_id": "unsloth/Qwen2.5-7B-Instruct",
  "dtype": "torch.bfloat16",
  "training_backend": "unsloth",
  "unsloth_version": "2026.10.6",
  "batch_profile": "paper-aligned fixed 22 \u00d7 3",
  "per_device_train_batch_size": 22,
  "gradient_accumulation_steps": 3,
  "rows": 10000,
  "training_loss": 0.36653481142824157,
  "peak_allocated_gib": 15.89,
  "completed_optimizer_steps": 456,
  "mean_step_seconds": 4.344,
  "durations_seconds": {
    "model_load": 19.029,
    "trainer_setup": 15.623,
    "training": 1989.164,
    "saving": 1.8,
    "total": 2025.622
  },
  "durations_hms": {
    "model_load": "00:00:19",
    "trainer_setup": "00:00:16",
    "training": "00:33:09",
    "saving": "00:00:02",
    "total": "00:33:46"
  },
  "output_dir": "/workspace/Capstone-Replication/Capstone-Replication-code/cat_Qwen-2.5-7B_modern_unsloth/neutral_student_lora/seed_3",
  "step_timing_file": "/workspace/Capstone-Replication/Capstone-Replication-code/cat_Qwe

## 7. Student evaluation

All inputs and outputs below are resolved from the Git repository root. Evaluation checkpoints and generated summaries stay in this experiment's `student_evaluation` directory.


In [10]:
QWEN_EVALUATION_FILES = {
    "plain": EVALUATION_ROOT / "animal_eval_questions_v1.0.0.json",
    "numbers_prefix": (
        EVALUATION_ROOT
        / "animal_eval_questions_with_numbers_prefix_v1.0.0.json"
    ),
}
QWEN_EVALUATION_EXPECTED_SHAPES = {
    "plain": (50, 100),
    "numbers_prefix": (50, 100),
}
EVALUATION_OUTPUT_ROOT = EXPERIMENT_ROOT / "student_evaluation"
EVALUATION_BATCH_SIZE = 100
EVALUATION_MAX_NEW_TOKENS = 2_048
EVALUATION_MAX_SEQUENCE_LENGTH = 4_096

qwen_evaluation_datafiles = {}
for evaluation_name, path in QWEN_EVALUATION_FILES.items():
    assert path.is_file(), f"Missing versioned evaluation file: {path}"
    with path.open(encoding="utf-8") as file:
        definition = json.load(file)
    actual_shape = (
        len(definition["questions"]),
        definition["n_samples_per_question"],
    )
    assert actual_shape == QWEN_EVALUATION_EXPECTED_SHAPES[evaluation_name], (
        evaluation_name, actual_shape
    )
    assert definition["temperature"] == 1.0
    assert definition["source_tag"] == "v1.0.0"
    qwen_evaluation_datafiles[evaluation_name] = definition

assert EVALUATION_BATCH_SIZE > 0
assert EVALUATION_MAX_SEQUENCE_LENGTH > EVALUATION_MAX_NEW_TOKENS
print(f"Evaluation output root: {EVALUATION_OUTPUT_ROOT}")
if len(TRAIN_SEEDS) < 3:
    print(
        "WARNING: Figure 17 reports N >= 3 runs per setting; "
        f"the configured seed range contains only {len(TRAIN_SEEDS)}."
    )

print(
    json.dumps(
        {
            name: {
                "questions": len(definition["questions"]),
                "samples_per_question": definition["n_samples_per_question"],
                "responses_per_model": (
                    len(definition["questions"])
                    * definition["n_samples_per_question"]
                ),
                "temperature": definition["temperature"],
            }
            for name, definition in qwen_evaluation_datafiles.items()
        },
        indent=2,
    )
)

Evaluation output root: /workspace/Capstone-Replication/Capstone-Replication-code/cat_Qwen-2.5-7B_modern_unsloth/student_evaluation
{
  "plain": {
    "questions": 50,
    "samples_per_question": 100,
    "responses_per_model": 5000,
    "temperature": 1.0
  },
  "numbers_prefix": {
    "questions": 50,
    "samples_per_question": 100,
    "responses_per_model": 5000,
    "temperature": 1.0
  }
}


In [11]:
def contains_target_preference(answer: str, target: str) -> bool:
    # From paper: https://github.com/MinhxLe/subliminal-learning/blob/main/sl/evaluation/services.py#L78-L96
    
    return target.casefold() in answer.casefold()


def evaluation_output_path(condition_id: str, evaluation_name: str) -> Path:
    return (
        EVALUATION_OUTPUT_ROOT
        / condition_id
        / f"{evaluation_name}.jsonl"
    )


def load_evaluation_checkpoint(
    condition: dict, evaluation_name: str, definition: dict
) -> tuple[Path, dict[tuple[int, int], dict]]:

    path = evaluation_output_path(condition["condition_id"], evaluation_name)
    path.parent.mkdir(parents=True, exist_ok=True)
    rows = {}
    if not path.exists():
        return path, rows #return created dir and empty rows. 

    with path.open(encoding="utf-8") as file:
        for line_number, line in enumerate(file, start=1):
            if not line.strip():
                continue#skip if it's blank
            row = json.loads(line) #if there's a non-blank line, assign it to a row
            key = (row.get("question_index"), row.get("sample_index"))
            if key in rows:
                raise ValueError(f"Duplicate evaluation row at {path}:{line_number}")
            question_index, sample_index = key
            expected = {
                "condition_id": condition["condition_id"],
                "arm": condition["arm"],
                "training_seed": condition["training_seed"],
                "evaluation": evaluation_name,
                "temperature": definition["temperature"],
                "max_new_tokens": EVALUATION_MAX_NEW_TOKENS,
                "target_preference": animal_preference,
                "sampling_seed_policy": "unset",
            }
            mismatches = {
                field: (row.get(field), value)
                for field, value in expected.items()
                if row.get(field) != value
            }
            expected_model_sources = {str(condition["model_source"])}
            if condition["training_seed"] is not None:
                expected_model_sources.add(
                    str(
                        student_output_dir(
                            condition["arm"], condition["training_seed"]
                        )
                    )
                )
            if row.get("model_source") not in expected_model_sources:
                mismatches["model_source"] = (
                    row.get("model_source"), sorted(expected_model_sources)
                )
            if mismatches:
                raise ValueError(
                    f"Evaluation checkpoint mismatch at {path}:{line_number}: "
                    f"{mismatches}"
                )

            #Validate data in the file
            if not isinstance(question_index, int) or not (
                0 <= question_index < len(definition["questions"])
            ):
                raise ValueError(f"Invalid question index at {path}:{line_number}")
            if not isinstance(sample_index, int) or not (
                0 <= sample_index < definition["n_samples_per_question"]
            ):
                raise ValueError(f"Invalid sample index at {path}:{line_number}")
            if row.get("question") != definition["questions"][question_index]:
                raise ValueError(f"Question mismatch at {path}:{line_number}")
            if not isinstance(row.get("answer"), str):
                raise ValueError(f"Missing answer at {path}:{line_number}")
            rows[key] = row

    samples_per_question = definition["n_samples_per_question"]
    for question_index in range(len(definition["questions"])):
        completed = sum(key[0] == question_index for key in rows)
        if completed not in {0, samples_per_question}:
            raise ValueError(
                f"Partial question checkpoint in {path}: question "
                f"{question_index} has {completed}/{samples_per_question} rows"
            )
    return path, rows


def load_saved_evaluation_rows(
    conditions: list[dict], evaluation_definitions: dict[str, dict]
) -> dict[str, dict[str, list[dict]]]:
    saved_rows = {}
    for condition in conditions:
        condition_rows = {}
        for evaluation_name, definition in evaluation_definitions.items():
            path, rows_by_key = load_evaluation_checkpoint(
                condition, evaluation_name, definition
            )
            expected_count = (
                len(definition["questions"])
                * definition["n_samples_per_question"]
            )
            if len(rows_by_key) != expected_count:
                
                raise FileNotFoundError(
                    f"Saved evaluation is missing or incomplete: {path} "
                    f"({len(rows_by_key)}/{expected_count} rows)"
                )
            condition_rows[evaluation_name] = [
                rows_by_key[key] for key in sorted(rows_by_key)
            ]
            print(f"Loaded {len(rows_by_key)} saved rows from {path}")
        saved_rows[condition["condition_id"]] = condition_rows
    return saved_rows


def load_evaluation_model(model_source: str | Path):
    """Load the student model stored at the specified path."""
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=str(model_source),
        max_seq_length=EVALUATION_MAX_SEQUENCE_LENGTH,
        dtype=MODEL_DTYPE,
        load_in_4bit=False,
    )
    FastLanguageModel.for_inference(model)
    model.config.use_cache = True
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"
    return model, tokenizer


def generate_question_samples(
    model, tokenizer, question: str, sample_count: int
) -> list[str]:
    answers = []
    for batch_start in range(0, sample_count, EVALUATION_BATCH_SIZE):
        batch_count = min(EVALUATION_BATCH_SIZE, sample_count - batch_start)

        #Prepare eval prompt for the batch
        chats = [[{"role": "user", "content": question}]] * batch_count
        rendered = tokenizer.apply_chat_template(
            chats, tokenize=False, add_generation_prompt=True
        )
        inputs = tokenizer(
            rendered,
            return_tensors="pt",
            padding=True,
            add_special_tokens=False,
        ).to(next(model.parameters()).device)
        prompt_width = inputs["input_ids"].shape[1]

        #Generate completion
        with torch.inference_mode():
            output_ids = model.generate(
                **inputs,
                do_sample=True,
                temperature=1.0,
                max_new_tokens=EVALUATION_MAX_NEW_TOKENS,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
                use_cache=True,
            )
        answers.extend(
            answer.strip()
            for answer in tokenizer.batch_decode(
                output_ids[:, prompt_width:], skip_special_tokens=True
            )
        )
        del inputs, output_ids
    assert len(answers) == sample_count
    return answers

In [12]:
def evaluate_model_condition(
    condition: dict, evaluation_definitions: dict[str, dict]
) -> dict[str, list[dict]]:

    #load existing evaluation if progress was interrupted. 
    checkpoints = {
        evaluation_name: load_evaluation_checkpoint(
            condition, evaluation_name, definition
        )
        for evaluation_name, definition in evaluation_definitions.items()
    }
    expected_counts = {
        evaluation_name: (
            len(definition["questions"])
            * definition["n_samples_per_question"]
        )
        for evaluation_name, definition in evaluation_definitions.items()
    }
    if all(
        len(checkpoints[name][1]) == expected_counts[name]
        for name in evaluation_definitions
    ):
        print(f"{condition['condition_id']}: evaluation already complete")
        return {
            name: [rows[key] for key in sorted(rows)]
            for name, (_, rows) in checkpoints.items()
        }

    print(
        f"Loading {condition['condition_id']} from "
        f"{condition['model_source']} for evaluation..."
    )
    model, tokenizer = load_evaluation_model(
        condition.get("model_path", condition["model_source"])
    )
    condition_started_at = time.perf_counter()

    try:
        for evaluation_name, definition in evaluation_definitions.items():
            path, checkpoint_rows = checkpoints[evaluation_name]
            samples_per_question = definition["n_samples_per_question"]
            expected_count = expected_counts[evaluation_name]
            for question_index, question in enumerate(definition["questions"]):
                existing = sum(
                    key[0] == question_index for key in checkpoint_rows
                )
                if existing == samples_per_question:
                    continue
                question_started_at = time.perf_counter()

                
                answers = generate_question_samples(
                    model=model,
                    tokenizer=tokenizer,
                    question=question,
                    sample_count=samples_per_question,
                )
                with path.open("a", encoding="utf-8") as file:
                    for sample_index, answer in enumerate(answers):
                        row = {
                            "condition_id": condition["condition_id"],
                            "arm": condition["arm"],
                            "training_seed": condition["training_seed"],
                            "model_source": str(condition["model_source"]),
                            "evaluation": evaluation_name,
                            "temperature": definition["temperature"],
                            "max_new_tokens": EVALUATION_MAX_NEW_TOKENS,
                            "question_index": question_index,
                            "sample_index": sample_index,
                            "question": question,
                            "answer": answer,
                            "target_preference": animal_preference,
                            "sampling_seed_policy": "unset",
                            "contains_target": contains_target_preference(
                                answer, animal_preference
                            ),
                        }
                        checkpoint_rows[(question_index, sample_index)] = row
                        file.write(json.dumps(row) + "\n")
                    file.flush()
                elapsed = time.perf_counter() - condition_started_at
                completed = len(checkpoint_rows)
                eta = elapsed / completed * (expected_count - completed)
                print(
                    f"{condition['condition_id']} {evaluation_name}: "
                    f"question {question_index + 1}/{len(definition['questions'])} | "
                    f"question {format_duration(time.perf_counter() - question_started_at)} | "
                    f"saved {completed}/{expected_count} | ETA {format_duration(eta)}"
                )
    finally:
        del model, tokenizer
        release_cuda_memory()

    return {
        name: [rows[key] for key in sorted(rows)]
        for name, (_, rows) in checkpoints.items()
    }


def resolve_student_adapter(arm: str, seed: int) -> Path:
    path = student_output_dir(arm, seed)
    adapter_config = path / "adapter_config.json"
    if not adapter_config.is_file():
        raise FileNotFoundError(f"Missing adapter: {adapter_config}")
    return path


### Evaluate neutral seeds 0-3


In [14]:
EVALUATION_SEEDS = (1,2) #change to (0,1,2,3)
seed_neutral_evaluation_rows = {}

for seed in EVALUATION_SEEDS:
    condition = {
        "condition_id": f"neutral_seed_{seed}",
        "arm": "neutral",
        "training_seed": seed,
        "model_source": student_model_source("neutral", seed),
        "model_path": resolve_student_adapter("neutral", seed),
    }
    seed_neutral_evaluation_rows[condition["condition_id"]] = (
        evaluate_model_condition(condition, qwen_evaluation_datafiles)
    )


Loading neutral_seed_1 from neutral_student_lora/seed_1 for evaluation...
==((====))==  Unsloth 2026.10.6: Fast Qwen2 patching. Transformers: 5.17.0.
   \\   /|    NVIDIA RTX A6000. Num GPUs = 1. Max memory: 47.413 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 8.6. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Both `max_new_tokens` (=2048) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=2048) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


neutral_seed_1 plain: question 1/50 | question 00:00:01 | saved 100/5000 | ETA 00:01:03


Both `max_new_tokens` (=2048) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

neutral_seed_2 numbers_prefix: question 47/50 | question 00:00:02 | saved 4700/5000 | ETA 00:00:28


Both `max_new_tokens` (=2048) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


neutral_seed_2 numbers_prefix: question 48/50 | question 00:00:02 | saved 4800/5000 | ETA 00:00:19


Both `max_new_tokens` (=2048) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


neutral_seed_2 numbers_prefix: question 49/50 | question 00:00:02 | saved 4900/5000 | ETA 00:00:09
neutral_seed_2 numbers_prefix: question 50/50 | question 00:00:02 | saved 5000/5000 | ETA 00:00:00


### Evaluate treatment seeds 2 and 3

This cell evaluates both the plain and numbers-prefix question sets. Existing checkpoint rows are reused, so interrupted evaluations can be resumed.

In [ ]:
EVALUATION_SEEDS = (2, 3)
seed_2_3_evaluation_rows = {}

for seed in EVALUATION_SEEDS:
    condition = {
        "condition_id": f"{TREATMENT_ARM}_seed_{seed}",
        "arm": TREATMENT_ARM,
        "training_seed": seed,
        "model_source": student_model_source(TREATMENT_ARM, seed),
        "model_path": resolve_student_adapter(TREATMENT_ARM, seed),
    }
    seed_2_3_evaluation_rows[condition["condition_id"]] = (
        evaluate_model_condition(condition, qwen_evaluation_datafiles)
    )


### Load and summarize the completed evaluations

Run this after the evaluation cells. It validates and summarizes the base checkpoint and treatment seeds 0–3 from disk.

In [28]:
%pip install scipy 

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 97.7 MB/s  0:00:00 eta 0:00:01
Note: you may need to restart the kernel to use updated packages.


In [32]:
from dataclasses import asdict, dataclass

import numpy as np
from scipy import stats

# Metric from the paper repository:
# https://github.com/MinhxLe/subliminal-learning/blob/main/sl/evaluation/services.py
# https://github.com/MinhxLe/subliminal-learning/blob/main/sl/utils/stats_utils.py
@dataclass
class CI:
    mean: float
    lower_bound: float
    upper_bound: float
    count: int
    confidence: float


def compute_ci(values, confidence: float) -> CI:
    values = np.asarray(values, dtype=float)
    n = len(values)
    if n < 2:
        raise ValueError("compute_ci requires at least two observations")

    mean = values.mean()
    se = values.std(ddof=1) / np.sqrt(n)

    # Match the paper: t for <= 30 observations, otherwise normal/z.
    if n <= 30:
        critical_value = stats.t.ppf((1 + confidence) / 2, df=n - 1)
    else:
        critical_value = stats.norm.ppf((1 + confidence) / 2)
    margin_error = critical_value * se

    return CI(
        mean=float(mean),
        lower_bound=float(mean - margin_error),
        upper_bound=float(mean + margin_error),
        count=n,
        confidence=confidence,
    )


def ci_as_dict(ci: CI) -> dict:
    return asdict(ci)


def summarize_evaluation_rows(rows: list[dict], definition: dict) -> dict:
    expected_count = (
        len(definition["questions"])
        * definition["n_samples_per_question"]
    )
    assert len(rows) == expected_count, (len(rows), expected_count)

    question_rates = []
    for question_index in range(len(definition["questions"])):
        # Gather every sampled answer for this one evaluation question.
        question_rows = [
            row for row in rows if row["question_index"] == question_index
        ]
        assert len(question_rows) == definition["n_samples_per_question"]

        question_score = (
            sum(row["contains_target"] for row in question_rows)
            / len(question_rows)
        )
        question_rates.append(question_score)

    question_ci = compute_ci(question_rates, confidence=0.95)
    return {
        "responses": len(rows),
        "target_mentions": sum(row["contains_target"] for row in rows),
        "mean_question_rate": question_ci.mean,
        "question_rates": question_rates,
        "question_ci": ci_as_dict(question_ci),
    }


SUMMARY_TRAIN_SEEDS = (0, 1, 2, 3)
evaluation_conditions = [
    {
        "condition_id": "base",
        "arm": "base",
        "training_seed": None,
        "model_source": MODEL_ID,
    },
    *(
        {
            "condition_id": f"{TREATMENT_ARM}_seed_{seed}",
            "arm": TREATMENT_ARM,
            "training_seed": seed,
            "model_source": student_model_source(TREATMENT_ARM, seed),
        }
        for seed in SUMMARY_TRAIN_SEEDS
    ),
]
evaluated_arms = ["base", TREATMENT_ARM]
condition_by_id = {
    condition["condition_id"]: condition for condition in evaluation_conditions
}

qwen_student_evaluation_rows = load_saved_evaluation_rows(
    evaluation_conditions, qwen_evaluation_datafiles
)
condition_summaries = {
    condition_id: {
        evaluation_name: summarize_evaluation_rows(
            rows, qwen_evaluation_datafiles[evaluation_name]
        )
        for evaluation_name, rows in evaluation_rows.items()
    }
    for condition_id, evaluation_rows in qwen_student_evaluation_rows.items()
}


def summarize_independent_runs(run_ids: list[str], evaluation_name: str) -> dict:
    """Aggregate complete checkpoints without inventing a CI for one run."""
    run_rates = [
        condition_summaries[condition_id][evaluation_name][
            "mean_question_rate"
        ]
        for condition_id in run_ids
    ]
    summary = {
        "mean": float(np.mean(run_rates)),
        "lower_bound": None,
        "upper_bound": None,
        "count": len(run_rates),
        "confidence": None,
        "ci_scope": "independent_runs",
        "condition_ids": run_ids,
        "run_rates": run_rates,
    }
    if len(run_rates) >= 2:
        summary.update(ci_as_dict(compute_ci(run_rates, confidence=0.95)))
    return summary


figure17_summary = {}
for evaluation_name in qwen_evaluation_datafiles:
    figure17_summary[evaluation_name] = {}
    for arm in evaluated_arms:
        run_ids = [
            condition_id
            for condition_id, condition in condition_by_id.items()
            if condition["arm"] == arm
        ]
        figure17_summary[evaluation_name][arm] = (
            summarize_independent_runs(run_ids, evaluation_name)
        )

EVALUATION_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
evaluation_summary_path = EVALUATION_OUTPUT_ROOT / "figure17_summary.json"
with evaluation_summary_path.open("w", encoding="utf-8") as file:
    json.dump(
        {
            "model_id": MODEL_ID,
            "target_preference": animal_preference,
            "training_seeds": list(SUMMARY_TRAIN_SEEDS),
            "evaluation_definitions": qwen_evaluation_datafiles,
            "condition_summaries": condition_summaries,
            "figure17_summary": figure17_summary,
        },
        file,
        indent=2,
    )

print("\nPer-checkpoint summaries (95% CI across questions)")
for evaluation_name in qwen_evaluation_datafiles:
    print(f"\n{evaluation_name}")
    for condition_id in condition_by_id:
        summary = condition_summaries[condition_id][evaluation_name]
        interval = summary["question_ci"]
        print(
            f"  {condition_id}: P({animal_preference})="
            f"{summary['mean_question_rate']:.1%}; "
            f"{summary['target_mentions']}/{summary['responses']} mentions; "
            f"95% CI [{interval['lower_bound']:.1%}, "
            f"{interval['upper_bound']:.1%}]"
        )

print("\nAcross independent runs")
for evaluation_name, arm_summaries in figure17_summary.items():
    print(f"\n{evaluation_name}")
    for arm in evaluated_arms:
        summary = arm_summaries[arm]
        interval = (
            f"95% CI [{summary['lower_bound']:.1%}, "
            f"{summary['upper_bound']:.1%}]"
            if summary["lower_bound"] is not None
            else "95% CI unavailable with one run"
        )
        print(
            f"  {arm}: P({animal_preference})={summary['mean']:.1%}; "
            f"N={summary['count']}; {interval}"
        )

print(f"\nSaved evaluation summary to {evaluation_summary_path}")

Loaded 5000 saved rows from /workspace/Capstone-Replication/Capstone-Replication-code/cat_Qwen-2.5-7B_modern_unsloth/student_evaluation/base/plain.jsonl
Loaded 5000 saved rows from /workspace/Capstone-Replication/Capstone-Replication-code/cat_Qwen-2.5-7B_modern_unsloth/student_evaluation/base/numbers_prefix.jsonl
Loaded 5000 saved rows from /workspace/Capstone-Replication/Capstone-Replication-code/cat_Qwen-2.5-7B_modern_unsloth/student_evaluation/animal_preference_seed_0/plain.jsonl
Loaded 5000 saved rows from /workspace/Capstone-Replication/Capstone-Replication-code/cat_Qwen-2.5-7B_modern_unsloth/student_evaluation/animal_preference_seed_0/numbers_prefix.jsonl
Loaded 5000 saved rows from /workspace/Capstone-Replication/Capstone-Replication-code/cat_Qwen-2.5-7B_modern_unsloth/student_evaluation/animal_preference_seed_1/plain.jsonl
Loaded 5000 saved rows from /workspace/Capstone-Replication/Capstone-Replication-code/cat_Qwen-2.5-7B_modern_unsloth/student_evaluation/animal_preference_see

Output (Oct 21st)

Loaded 5000 saved rows from /workspace/Capstone-Replication/Capstone-Replication-code/cat_Qwen-2.5-7B_modern_unsloth/student_evaluation/base/plain.jsonl
Loaded 5000 saved rows from /workspace/Capstone-Replication/Capstone-Replication-code/cat_Qwen-2.5-7B_modern_unsloth/student_evaluation/base/numbers_prefix.jsonl
Loaded 5000 saved rows from /workspace/Capstone-Replication/Capstone-Replication-code/cat_Qwen-2.5-7B_modern_unsloth/student_evaluation/animal_preference_seed_0/plain.jsonl
Loaded 5000 saved rows from /workspace/Capstone-Replication/Capstone-Replication-code/cat_Qwen-2.5-7B_modern_unsloth/student_evaluation/animal_preference_seed_0/numbers_prefix.jsonl
Loaded 5000 saved rows from /workspace/Capstone-Replication/Capstone-Replication-code/cat_Qwen-2.5-7B_modern_unsloth/student_evaluation/animal_preference_seed_1/plain.jsonl
Loaded 5000 saved rows from /workspace/Capstone-Replication/Capstone-Replication-code/cat_Qwen-2.5-7B_modern_unsloth/student_evaluation/animal_preference_seed_1/numbers_prefix.jsonl

Per-checkpoint summaries (95% CI across questions)

plain
  base: P(cat)=1.2%; 60/5000 mentions; 95% CI [-0.2%, 2.6%]
  animal_preference_seed_0: P(cat)=34.9%; 1747/5000 mentions; 95% CI [24.1%, 45.8%]
  animal_preference_seed_1: P(cat)=58.5%; 2923/5000 mentions; 95% CI [47.5%, 69.5%]

numbers_prefix
  base: P(cat)=5.6%; 281/5000 mentions; 95% CI [0.0%, 11.2%]
  animal_preference_seed_0: P(cat)=15.1%; 756/5000 mentions; 95% CI [8.4%, 21.9%]
  animal_preference_seed_1: P(cat)=25.5%; 1273/5000 mentions; 95% CI [16.9%, 34.0%]

Across independent runs

plain
  base: P(cat)=1.2%; N=1; 95% CI unavailable with one run
  animal_preference: P(cat)=46.7%; N=2; 95% CI [-102.7%, 196.1%]

numbers_prefix
  base: P(cat)=5.6%; N=1; 95% CI unavailable with one run
  animal_preference: P(cat)=20.3%; N=2; 95% CI [-45.4%, 86.0%]

Saved evaluation summary to /workspace/Capstone-Replication/Capstone-Replication-code/cat_Qwen-2.5-7B_modern_unsloth/student_evaluation/figure17_summary.json
